# Blog figures and model inspection for the explainability report

This notebook is the single place where the static visual assets for the HTML report are refreshed. It performs two tasks:

1. Run a real forward inspection of the trained ResNet50 on one AwA2 image and save the `resnet-forward.png` asset used in the model section.
2. Export the already computed attribution, stress-test, concept and bottleneck figures into `docs/assets/xai-report/` and generate the lightweight summary graphics used by the report.

The notebook does not retrain the classifier and does not recompute the expensive attribution audits. It expects the relevant checkpoints, CSV files and figure outputs to already exist.


## Real ResNet50 forward inspection

This section inspects what happens during one real prediction of the trained ResNet50 baseline. It uses the project DataLoader, the saved checkpoint and forward hooks on the actual PyTorch model.

The goal is not to simulate the network. The goal is to observe the real intermediate tensors used by the classifier: input tensor, convolutional stages, residual layers, logits, softmax probabilities and the Grad-CAM target layer. The saved figure becomes the static ResNet forward-pass figure in the HTML report.


In [ ]:
from pathlib import Path
import os
import tempfile

_CACHE_ROOT = Path(tempfile.gettempdir()) / 'deep_learning_xai'
_MPLCONFIGDIR = _CACHE_ROOT / 'matplotlib'
_XDG_CACHE_HOME = _CACHE_ROOT / 'xdg-cache'
_MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
_XDG_CACHE_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLBACKEND', 'Agg')
os.environ.setdefault('MPLCONFIGDIR', str(_MPLCONFIGDIR))
os.environ.setdefault('XDG_CACHE_HOME', str(_XDG_CACHE_HOME))

from IPython.display import Image, display
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src").exists() and (candidate / "scripts").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate project root containing src/ and scripts/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MANIFEST = PROJECT_ROOT / "data" / "AWA2_subset_background20" / "awa2_manifest_subset.csv"
CHECKPOINT = PROJECT_ROOT / "outputs" / "checkpoints" / "best_resnet50_awa2.pt"
OUTPUT = PROJECT_ROOT / "outputs" / "figures" / "real_forward_inspection_notebook.png"

print("project_root:", PROJECT_ROOT)
print("manifest:", MANIFEST, "exists=", MANIFEST.exists())
print("checkpoint:", CHECKPOINT, "exists=", CHECKPOINT.exists())

## Load one real sample

The selected sample comes from the same manifest used by the project. The transform is the standard ResNet preprocessing: resize, center crop, tensor conversion and ImageNet normalization.

In [ ]:
from src.data import ImageManifestDataset, build_resnet_transforms, infer_num_classes, load_class_names
from src.model import build_resnet50_classifier, get_device, load_checkpoint
from src.forward_inspection import ForwardActivationInspector, print_trace_summary, save_prediction_trace_figure
from src.utils import set_seed

set_seed(42)
device = get_device("auto")
num_classes = infer_num_classes(MANIFEST)
class_names = load_class_names(MANIFEST)

dataset = ImageManifestDataset(
    manifest_path=MANIFEST,
    split="test",
    transform=build_resnet_transforms(train=False),
)

SAMPLE_INDEX = 0
image, label, true_name, image_path = dataset[SAMPLE_INDEX]
print("device:", device)
print("num_classes:", num_classes)
print("image:", image_path)
print("true label:", int(label), true_name)
print("input tensor shape:", tuple(image.shape))

## Run the real model and capture intermediate tensors

Forward hooks are attached to `conv1`, `maxpool`, `layer1`, `layer2`, `layer3`, `layer4` and `avgpool`. The notebook then performs one real forward pass and prints tensor statistics. The Grad-CAM map is computed on `layer4[-1]`, the last residual block.

In [ ]:
model = build_resnet50_classifier(num_classes=num_classes, pretrained=False).to(device)
load_checkpoint(model, CHECKPOINT, device)
model.eval()

with ForwardActivationInspector(model) as inspector:
    trace = inspector.run(image=image, target_label=None, compute_gradcam=True)

print_trace_summary(trace, class_names)

## Visual interpretation of the real forward pass

The figure shows the denormalized input image, activation summaries for the main ResNet stages, the Grad-CAM map, top softmax probabilities and a table of tensor statistics. The activation maps are not new model outputs: they are channel-reduced visual summaries of the real intermediate tensors.

In [ ]:
save_prediction_trace_figure(
    image=image,
    trace=trace,
    class_names=class_names,
    output_path=OUTPUT,
    true_label=int(label),
    image_path=image_path,
)


display(Image(filename=str(OUTPUT)))

## How to interpret this inspection

- `conv1` and `maxpool` show early visual primitives, such as color transitions, contours and texture.
- `layer1` and `layer2` compress local patterns into more structured visual parts.
- `layer3` and `layer4` are the fine-tuned high-level blocks in this project.
- `layer4` has low spatial resolution, so Grad-CAM starts from coarse evidence and is upsampled for display.
- The softmax probability reports confidence, but the attribution map should be read as evidence for the selected target score, not as proof that the model used the correct semantic reason.

## Export report assets

This section copies the generated project figures into the tracked report asset directory and creates compact summary figures from the available CSV files. The forward-pass figure created above is exported together with the other report figures.


In [ ]:

from pathlib import Path
import os
import tempfile

_CACHE_ROOT = Path(tempfile.gettempdir()) / 'deep_learning_xai'
_MPLCONFIGDIR = _CACHE_ROOT / 'matplotlib'
_XDG_CACHE_HOME = _CACHE_ROOT / 'xdg-cache'
_MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
_XDG_CACHE_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLBACKEND', 'Agg')
os.environ.setdefault('MPLCONFIGDIR', str(_MPLCONFIGDIR))
os.environ.setdefault('XDG_CACHE_HOME', str(_XDG_CACHE_HOME))

import json
import re
import shutil
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

FIG_DIR = PROJECT_ROOT / 'outputs' / 'figures'
DOCS_DIR = PROJECT_ROOT / 'docs'
REPORT_DIR = PROJECT_ROOT / 'outputs' / 'reports'
DOC_ASSET_DIR = DOCS_DIR / 'assets' / 'xai-report'
XAI_EXAMPLE_FIG = FIG_DIR / 'xai_method_comparison_notebook.png'
XAI_INCORRECT_FIG = FIG_DIR / 'xai_incorrect_examples_notebook.png'
FORWARD_FIG = FIG_DIR / 'real_forward_inspection_notebook.png'
PERTURBATION_FIG = FIG_DIR / 'phase5_perturbations_notebook.png'
SALIENCY_COMPARISON_FIG = FIG_DIR / 'phase5_saliency_comparison_notebook.png'
STRESS_FIGURES = {
    'gradcam': FIG_DIR / 'phase5_saliency_comparison_notebook_gradcam.png',
    'integrated_gradients': FIG_DIR / 'phase5_saliency_comparison_notebook_integrated_gradients.png',
}
ADVANCED_FIGURE_DIR = FIG_DIR / 'advanced_attribution_audit_notebook'
FAITHFULNESS_FIGURES = {
    'gradcam': ADVANCED_FIGURE_DIR / 'gradcam_deletion_insertion.png',
    'integrated_gradients': ADVANCED_FIGURE_DIR / 'integrated_gradients_deletion_insertion.png',
}
MISCLASSIFICATION_FIGURE_DIR = FIG_DIR / 'misclassification_audit'
MISCLASSIFICATION_FIGURES = {
    'error-target-contrast-gradcam.png': MISCLASSIFICATION_FIGURE_DIR / 'target_contrast_gradcam.png',
    'error-target-contrast-ig.png': MISCLASSIFICATION_FIGURE_DIR / 'target_contrast_integrated_gradients.png',
    'error-perturbation-margins.png': MISCLASSIFICATION_FIGURE_DIR / 'perturbation_decision_margins.png',
    'error-deletion-curves.png': MISCLASSIFICATION_FIGURE_DIR / 'deletion_curves.png',
    'error-cbm-concepts.png': MISCLASSIFICATION_FIGURE_DIR / 'cbm_concept_evidence.png',
}
CBM_CASE_FIGURES = {
    'cbm-real-error-case-1.png': FIG_DIR / 'phase8_cbm_real_error_case_1.png',
    'cbm-real-error-case-2.png': FIG_DIR / 'phase8_cbm_real_error_case_2.png',
}
TCAV_SPURIOUS_FIGURES = {
    'tcav-spurious-pairs.png': FIG_DIR / 'phase7_spurious_tcav_pairs.png',
    'tcav-bridge-concepts.png': FIG_DIR / 'phase7_tcav_bridge_concepts.png',
}
FIG_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)
DOC_ASSET_DIR.mkdir(parents=True, exist_ok=True)
BLOG_FIGURE_EXPORTS = {
    'correct-attributions.png': XAI_EXAMPLE_FIG,
    'incorrect-attributions.png': XAI_INCORRECT_FIG,
    'resnet-forward.png': FORWARD_FIG,
    'background-perturbations.png': PERTURBATION_FIG,
    'saliency-stress-comparison.png': SALIENCY_COMPARISON_FIG,
    'stress-gradcam.png': STRESS_FIGURES['gradcam'],
    'stress-integrated-gradients.png': STRESS_FIGURES['integrated_gradients'],
    'advanced-gradcam-deletion.png': FAITHFULNESS_FIGURES['gradcam'],
    'advanced-ig-deletion.png': FAITHFULNESS_FIGURES['integrated_gradients'],
    **MISCLASSIFICATION_FIGURES,
    **CBM_CASE_FIGURES,
    **TCAV_SPURIOUS_FIGURES,
}
missing_blog_assets = []
for destination_name, source_path in BLOG_FIGURE_EXPORTS.items():
    destination_path = DOC_ASSET_DIR / destination_name
    if source_path.exists():
        shutil.copy2(source_path, destination_path)
        print(f'copied {source_path} -> {destination_path}')
    elif destination_path.exists():
        print(f'preserved tracked asset; local source is absent: {destination_path}')
    else:
        missing_blog_assets.append((source_path, destination_path))

if missing_blog_assets:
    raise FileNotFoundError(
        'Each report asset requires either a current output source or an existing tracked destination: '
        f'{missing_blog_assets}'
    )

plt.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 180,
    'font.size': 10,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

PALETTE = {
    'navy': '#18324a',
    'teal': '#2a9d8f',
    'amber': '#e9a23b',
    'red': '#d95f5f',
    'blue': '#4f7cac',
    'soft': '#f4f8f7',
    'line': '#cfd8dc',
    'ink': '#263238',
    'muted': '#61717a',
}

def save_status_figure(path, title, message):
    fig, ax = plt.subplots(figsize=(9, 4.8))
    fig.patch.set_facecolor('white')
    ax.axis('off')
    ax.text(0.5, 0.62, title, transform=ax.transAxes, ha='center', va='center', fontsize=20, fontweight='bold', color=PALETTE['navy'])
    ax.text(0.5, 0.38, message, transform=ax.transAxes, ha='center', va='center', fontsize=12, color=PALETTE['muted'], wrap=True)
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    print(f'saved status figure {path}')

advanced = pd.read_csv(REPORT_DIR / 'advanced_attribution_audit_notebook_summary.csv')
PHASE5_CSV = REPORT_DIR / 'phase5_saliency_metrics_notebook.csv'
stress_metrics = pd.read_csv(PHASE5_CSV)
IOU_COLUMN = next((column for column in stress_metrics if column.startswith('iou_top_')), None)
if IOU_COLUMN is None:
    raise ValueError('Stress metrics CSV is missing an iou_top_* column.')
required_confidence_columns = {'original_target_probability', 'perturbed_target_probability', 'confidence_delta', 'confidence_drop'}
missing_confidence_columns = required_confidence_columns.difference(stress_metrics.columns)
if missing_confidence_columns:
    raise ValueError(f'Regenerate the stress metrics CSV; missing corrected confidence columns: {sorted(missing_confidence_columns)}')
for column in ['original_confidence', 'perturbed_confidence', *sorted(required_confidence_columns), IOU_COLUMN, 'spearman']:
    stress_metrics[column] = pd.to_numeric(stress_metrics[column], errors='coerce')
stress_metrics['prediction_changed'] = stress_metrics['prediction_changed'].astype(str).str.lower().eq('true')
stress_metrics['saliency_drift'] = (1.0 - stress_metrics[IOU_COLUMN]) + (1.0 - stress_metrics['spearman']) / 2.0
stress_metrics['prediction_transition'] = stress_metrics['original_prediction'].astype(str) + ' -> ' + stress_metrics['perturbed_prediction'].astype(str)
stress = (
    stress_metrics.groupby(['xai_method', 'perturbation'], as_index=False)
    .agg(
        prediction_change_rate=('prediction_changed', 'mean'),
        mean_iou=(IOU_COLUMN, 'mean'),
    )
)
unstable = stress_metrics.sort_values('saliency_drift', ascending=False).head(12)
method_names = {'gradcam': 'Grad-CAM', 'integrated_gradients': 'Integrated\nGradients'}
maintained_methods = list(method_names)
advanced = advanced[advanced['method'].isin(maintained_methods)].copy()
advanced['label'] = advanced['method'].map(method_names)
fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.2))
fig.patch.set_facecolor('white')

ax = axes[0, 0]
y = np.arange(len(advanced))
ax.barh(y, advanced['mean_animal_saliency_ratio'], color=PALETTE['teal'], label='animal')
ax.barh(y, advanced['mean_background_saliency_ratio'], left=advanced['mean_animal_saliency_ratio'], color=PALETTE['amber'], label='background')
ax.set_yticks(y, advanced['label'])
ax.set_xlim(0, 1)
ax.set_title('Where the saliency mass is located', pad=24)
ax.set_xlabel('fraction of total saliency')
for i, row in advanced.iterrows():
    ax.text(row['mean_animal_saliency_ratio'] / 2, i, f"{row['mean_animal_saliency_ratio']:.2f}", va='center', ha='center', color='white', fontweight='bold')
    ax.text(row['mean_animal_saliency_ratio'] + row['mean_background_saliency_ratio'] / 2, i, f"{row['mean_background_saliency_ratio']:.2f}", va='center', ha='center', color=PALETTE['navy'], fontweight='bold')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.01), ncol=2, frameon=False)
ax.grid(axis='x', alpha=0.18)

ax = axes[0, 1]
x = np.arange(len(advanced))
w = 0.38
ax.bar(x - w/2, advanced['mean_sensitivity_iou_top20'], width=w, color=PALETTE['blue'], label='IoU')
ax.bar(x + w/2, advanced['mean_sensitivity_spearman'], width=w, color=PALETTE['teal'], label='Spearman')
ax.set_xticks(x, advanced['label'])
ax.set_ylim(0, 1)
ax.set_title('Sensitivity stability under input noise')
ax.set_ylabel('higher is more stable')
for xpos, vals in [(x - w/2, advanced['mean_sensitivity_iou_top20']), (x + w/2, advanced['mean_sensitivity_spearman'])]:
    for xx, val in zip(xpos, vals):
        ax.text(xx, val + 0.025, f'{val:.2f}', ha='center', va='bottom', fontsize=8)
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.18)

ax = axes[1, 0]
ax.bar(x - w/2, advanced['mean_deletion_auc'], width=w, color=PALETTE['red'], label='Deletion AUC')
ax.bar(x + w/2, advanced['mean_insertion_auc'], width=w, color=PALETTE['teal'], label='Insertion AUC')
ax.set_xticks(x, advanced['label'])
ax.set_ylim(0, max(advanced['mean_insertion_auc'].max(), advanced['mean_deletion_auc'].max()) * 1.35)
ax.set_title('Behavioral faithfulness curves')
ax.set_ylabel('AUC')
for xpos, vals in [(x - w/2, advanced['mean_deletion_auc']), (x + w/2, advanced['mean_insertion_auc'])]:
    for xx, val in zip(xpos, vals):
        ax.text(xx, val + 0.01, f'{val:.2f}', ha='center', va='bottom', fontsize=8)
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.18)

ax = axes[1, 1]
bs = stress[stress['perturbation'].eq('background_swap') & stress['xai_method'].isin(maintained_methods)].copy()
bs['label'] = bs['xai_method'].map(method_names)
xx = np.arange(len(bs))
ax.bar(xx - w/2, bs['mean_iou'], width=w, color=PALETTE['blue'], label='background-swap IoU')
ax.bar(xx + w/2, bs['prediction_change_rate'], width=w, color=PALETTE['amber'], label='prediction change rate')
ax.set_xticks(xx, bs['label'])
ax.set_ylim(0, 1)
ax.set_title('Background-swap stress test')
ax.set_ylabel('rate / overlap')
for xpos, vals in [(xx - w/2, bs['mean_iou']), (xx + w/2, bs['prediction_change_rate'])]:
    for xxv, val in zip(xpos, vals):
        ax.text(xxv, val + 0.025, f'{val:.2f}', ha='center', va='bottom', fontsize=8)
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.18)

for ax in axes.ravel():
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines['left'].set_color('#d7dee2')
    ax.spines['bottom'].set_color('#d7dee2')

fig.suptitle('Quantitative evidence: saliency location, stability and faithfulness', fontsize=15, fontweight='bold', color=PALETTE['navy'])
fig.tight_layout(rect=[0, 0, 1, 0.95])
out = FIG_DIR / 'blog_metric_summary.png'
fig.savefig(out, bbox_inches='tight')
shutil.copy2(out, DOC_ASSET_DIR / 'metric-summary.png')
plt.close(fig)
print(f'saved {out}')

case_pool = stress_metrics[stress_metrics['perturbation'].eq('background_swap')].copy()
changed_pool = case_pool[case_pool['prediction_changed']].copy()
case_has_transition = not changed_pool.empty
selection_pool = changed_pool if case_has_transition else case_pool
case_index = selection_pool.groupby('index')['saliency_drift'].mean().idxmax()
case = case_pool[case_pool['index'].eq(case_index) & case_pool['xai_method'].isin(['gradcam', 'integrated_gradients'])].copy()
if case.empty:
    case = selection_pool[selection_pool['index'].eq(case_index)].head(2).copy()
case['label'] = case['xai_method'].map({'gradcam': 'Grad-CAM', 'integrated_gradients': 'IG'}).fillna(case['xai_method'])
base = case.iloc[0]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), gridspec_kw={'width_ratios': [1.1, 1, 1.2]})
fig.patch.set_facecolor('white')

ax = axes[0]
ax.axis('off')
ax.set_title('Single unstable example', color=PALETTE['navy'], fontweight='bold')
summary = [
    ('true class', str(base['true_class'])),
    ('original prediction', f"{base['original_prediction']} ({base['original_confidence']:.2f})"),
    ('after background swap', f"{base['perturbed_prediction']} ({base['perturbed_confidence']:.2f})"),
    ('transition', str(base['prediction_transition'])),
]
for i, (k, v) in enumerate(summary):
    y0 = 0.82 - i * 0.19
    patch = FancyBboxPatch((0.02, y0 - 0.075), 0.96, 0.13, boxstyle='round,pad=0.018,rounding_size=0.02', transform=ax.transAxes, fc=PALETTE['soft'], ec=PALETTE['line'])
    ax.add_patch(patch)
    ax.text(0.07, y0 + 0.018, k.upper(), transform=ax.transAxes, fontsize=8, color=PALETTE['muted'], fontweight='bold')
    ax.text(0.07, y0 - 0.035, v, transform=ax.transAxes, fontsize=11, color=PALETTE['ink'], fontweight='bold')

ax = axes[1]
vals = [base['original_target_probability'], base['perturbed_target_probability']]
ax.bar(['original', 'perturbed'], vals, color=[PALETTE['teal'], PALETTE['amber']])
ax.set_ylim(0, 1)
ax.set_title('Confidence drop')
ax.set_ylabel('fixed saliency-target probability')
for i, val in enumerate(vals):
    ax.text(i, val + 0.03, f'{val:.2f}', ha='center', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.18)

ax = axes[2]
labels = list(case['label'])
xpos = np.arange(len(case))
ax.bar(xpos - 0.18, case[IOU_COLUMN], width=0.36, color=PALETTE['blue'], label='Top-20 IoU')
ax.bar(xpos + 0.18, case['spearman'], width=0.36, color=PALETTE['teal'], label='Spearman')
ax.set_xticks(xpos, labels)
ax.set_ylim(0, 1)
ax.set_title('Explanation stability after background swap')
ax.set_ylabel('higher is more stable')
for xxv, val in zip(xpos - 0.18, case[IOU_COLUMN]):
    ax.text(xxv, val + 0.03, f'{val:.2f}', ha='center', fontsize=9)
for xxv, val in zip(xpos + 0.18, case['spearman']):
    ax.text(xxv, val + 0.03, f'{val:.2f}', ha='center', fontsize=9)
ax.legend(frameon=False, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.18)

case_title = ('Case study: a background change switched the class and moved the explanation' if case_has_transition else 'Case study: a background change moved the explanation')
fig.suptitle(case_title, fontsize=15, fontweight='bold', color=PALETTE['navy'])
fig.tight_layout(rect=[0, 0, 1, 0.92])
out = FIG_DIR / 'blog_case_study_summary.png'
fig.savefig(out, bbox_inches='tight')
shutil.copy2(out, DOC_ASSET_DIR / 'case-summary.png')
plt.close(fig)
print(f'saved {out}')

TCAV_CSV = REPORT_DIR / 'phase7_tcav_scores_notebook.csv'
CBM_SUMMARY_CSV = REPORT_DIR / 'phase8_cbm_summary_notebook.csv'
IMAGE_INTERVENTION_CSV = REPORT_DIR / 'phase8_image_concept_interventions_notebook.csv'
TCAV_HEATMAP_FIG = FIG_DIR / 'phase7_tcav_heatmap_notebook.png'
TCAV_EFFECT_FIG = FIG_DIR / 'phase7_tcav_top_scores_notebook.png'
CBM_SUMMARY_FIG = FIG_DIR / 'phase8_cbm_summary_notebook.png'
IMAGE_INTERVENTION_FIG = FIG_DIR / 'phase8_image_concept_interventions_notebook.png'
required_cbm_outputs = [CBM_SUMMARY_CSV, IMAGE_INTERVENTION_CSV, CBM_SUMMARY_FIG, IMAGE_INTERVENTION_FIG]
missing_cbm_outputs = [path for path in required_cbm_outputs if not path.exists()]
def export_concept_assets():

    tcav_available = TCAV_CSV.exists()
    tcav = pd.read_csv(TCAV_CSV) if tcav_available else pd.DataFrame()
    cbm = pd.read_csv(CBM_SUMMARY_CSV).iloc[0]
    image_interventions = pd.read_csv(IMAGE_INTERVENTION_CSV)
    validated_tcav_columns = {
        'concept', 'target_class', 'effect_size', 'tcav_std',
        'cav_validation_accuracy_mean', 'random_cav_validation_accuracy_mean',
        'p_value_corrected', 'significant',
    }
    legacy_tcav_columns = {'concept', 'target_class', 'tcav_score', 'cav_train_accuracy'}
    tcav_is_validated = tcav_available and validated_tcav_columns.issubset(tcav.columns)

    if tcav_is_validated:
        numeric_columns = [
            'effect_size', 'tcav_std', 'cav_validation_accuracy_mean',
            'random_cav_validation_accuracy_mean', 'p_value_corrected',
        ]
        for column in numeric_columns:
            tcav[column] = pd.to_numeric(tcav[column], errors='raise')
        tcav['significant_bool'] = tcav['significant'].astype(str).str.lower().eq('true')
        selected_tcav = tcav.sort_values(
            ['significant_bool', 'effect_size'], ascending=[False, False]
        ).head(10).copy()
    elif tcav_available and legacy_tcav_columns.issubset(tcav.columns):
        missing_legacy = legacy_tcav_columns.difference(tcav.columns)
        if missing_legacy:
            raise ValueError(f'TCAV CSV has an unsupported schema. Missing: {sorted(missing_legacy)}')
        tcav['tcav_score'] = pd.to_numeric(tcav['tcav_score'], errors='raise')
        tcav['cav_train_accuracy'] = pd.to_numeric(tcav['cav_train_accuracy'], errors='raise')
        selected_tcav = tcav.sort_values('tcav_score', ascending=False).head(10).copy()
        print(
            'TCAV note: the available CSV is exploratory legacy output. The figure reports '
            'raw TCAV scores and CAV training accuracy only; it does not claim held-out '
            'validation, random-control effects, variability, or statistical significance.'
        )
    else:
        tcav_available = False
        selected_tcav = pd.DataFrame({
            'concept': ['not available'],
            'target_class': [''],
            'tcav_score': [0.0],
            'cav_train_accuracy': [0.0],
        })
        print('TCAV outputs are unavailable; exporting explicit status panels without TCAV claims.')
    selected_tcav['label'] = selected_tcav['concept'].astype(str) + ' -> ' + selected_tcav['target_class'].astype(str)

    fig, axes = plt.subplots(1, 2, figsize=(13.6, 6.4), gridspec_kw={'width_ratios': [1.35, 1.0]})
    fig.patch.set_facecolor('white')
    ax = axes[0]
    positions = np.arange(len(selected_tcav))
    if tcav_is_validated:
        colors = [PALETTE['teal'] if value else '#9aa7ad' for value in selected_tcav['significant_bool']]
        ax.barh(positions, selected_tcav['effect_size'], xerr=selected_tcav['tcav_std'], color=colors)
    else:
        ax.barh(positions, selected_tcav['tcav_score'], color=PALETTE['teal'])
    ax.set_yticks(positions, selected_tcav['label'])
    ax.invert_yaxis()
    if tcav_is_validated:
        ax.axvline(0, color=PALETTE['ink'], linewidth=0.8)
        ax.set_title('TCAV effect beyond random CAVs')
        ax.set_xlabel('mean real minus random score')
    else:
        ax.set_xlim(0, 1)
        ax.set_title('Exploratory TCAV scores')
        ax.set_xlabel('fraction of positive directional derivatives')
    ax.grid(axis='x', alpha=0.18)

    ax = axes[1]
    x_tcav = np.arange(len(selected_tcav))
    if tcav_is_validated:
        width = 0.38
        ax.bar(x_tcav - width/2, selected_tcav['cav_validation_accuracy_mean'], width=width, color=PALETTE['teal'], label='real CAV')
        ax.bar(x_tcav + width/2, selected_tcav['random_cav_validation_accuracy_mean'], width=width, color=PALETTE['amber'], label='random CAV')
    else:
        ax.bar(x_tcav, selected_tcav['cav_train_accuracy'], color=PALETTE['amber'])
    ax.set_xticks(x_tcav, [str(value) for value in selected_tcav['concept']], rotation=55, ha='right')
    ax.set_ylim(0, 1)
    ax.set_title(
        'Held-out CAV validation'
        if tcav_is_validated
        else 'CAV fit on training activations\n(no held-out validation)'
    )
    ax.set_ylabel('accuracy')
    if tcav_is_validated:
        ax.legend(frameon=False)
    else:
        ax.set_xlabel('Training fit is not evidence of CAV generalization')
    ax.grid(axis='y', alpha=0.18)

    if not tcav_available:
        for status_ax, title, message in [
            (axes[0], 'TCAV results not available', 'No TCAV score CSV is present in the clean output state.'),
            (axes[1], 'CAV validation not available', 'No held-out CAV or random-control statistics are reported.'),
        ]:
            status_ax.clear()
            status_ax.axis('off')
            status_ax.text(0.5, 0.58, title, transform=status_ax.transAxes, ha='center', va='center', fontsize=14, fontweight='bold', color=PALETTE['navy'])
            status_ax.text(0.5, 0.40, message, transform=status_ax.transAxes, ha='center', va='center', color=PALETTE['muted'], wrap=True)

    for ax in axes:
        ax.spines[['top', 'right']].set_visible(False)
    if tcav_is_validated:
        concept_figure_title = 'Validated TCAV concept evidence: CAV generalization and random controls'
    elif tcav_available:
        concept_figure_title = 'TCAV concept evidence: exploratory CAV output'
    else:
        concept_figure_title = 'TCAV concept evidence: TCAV status explicitly unavailable'
    fig.suptitle(concept_figure_title, fontsize=15, fontweight='bold', color=PALETTE['navy'])
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    out = FIG_DIR / 'blog_concept_validation_summary.png'
    fig.savefig(out, bbox_inches='tight')
    shutil.copy2(out, DOC_ASSET_DIR / 'concept-validation-summary.png')
    if TCAV_HEATMAP_FIG.exists():
        shutil.copy2(TCAV_HEATMAP_FIG, DOC_ASSET_DIR / 'tcav-heatmap.png')
    else:
        save_status_figure(DOC_ASSET_DIR / 'tcav-heatmap.png', 'TCAV heatmap not available', 'The current clean output state does not contain TCAV scores.')
    if TCAV_EFFECT_FIG.exists():
        shutil.copy2(TCAV_EFFECT_FIG, DOC_ASSET_DIR / 'tcav-top-scores.png')
    else:
        save_status_figure(DOC_ASSET_DIR / 'tcav-top-scores.png', 'TCAV effect plot not available', 'No validated or exploratory TCAV effect file is being reported.')
    shutil.copy2(CBM_SUMMARY_FIG, DOC_ASSET_DIR / 'cbm-summary.png')
    shutil.copy2(IMAGE_INTERVENTION_FIG, DOC_ASSET_DIR / 'concept-interventions.png')
    plt.close(fig)
    print(f'saved {out}')

if missing_cbm_outputs:
    print(
        'Concept outputs are not available in the local outputs/ directory; preserving the existing tracked '
        'report assets. This is expected after pulling another clone because outputs/ is Git-ignored.'
    )
    print(f'Missing local sources: {missing_cbm_outputs}')
else:
    export_concept_assets()

HTML_REPORT = DOCS_DIR / 'intuitive_explainability.html'
html = HTML_REPORT.read_text(encoding='utf-8')
image_sources = re.findall(r'<img\s+[^>]*src="([^"]+)"', html, flags=re.IGNORECASE)
local_image_paths = [
    (HTML_REPORT.parent / source).resolve()
    for source in image_sources
    if not source.startswith(('http://', 'https://', 'data:'))
]
missing_report_images = [
    path for path in local_image_paths
    if not path.is_file() or path.stat().st_size == 0
]
if missing_report_images:
    raise FileNotFoundError(
        f'Blog export is incomplete; HTML image assets are missing or empty: {missing_report_images}'
    )
print(f'validated {len(local_image_paths)} local HTML image references')


In [ ]:

import subprocess
import sys

stress_dashboard_cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "experiments" / "plot_stress_metric_dashboard.py"),
    "--metrics-csv",
    str(REPORT_DIR / "phase5_saliency_metrics_notebook.csv"),
    "--advanced-summary-csv",
    str(REPORT_DIR / "advanced_attribution_audit_notebook_summary.csv"),
    "--advanced-detail-csv",
    str(REPORT_DIR / "advanced_attribution_audit_notebook.csv"),
    "--output-dir",
    str(FIG_DIR),
    "--docs-output-dir",
    str(DOC_ASSET_DIR),
    "--summary-output",
    str(REPORT_DIR / "stress_metric_dashboard_summary.csv"),
]
subprocess.run(stress_dashboard_cmd, cwd=PROJECT_ROOT, check=True)


In [ ]:

stress_curve_cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "experiments" / "run_stress_deletion_insertion_curves.py"),
    "--manifest",
    str(PROJECT_ROOT / "data" / "AWA2_subset_background20" / "awa2_manifest_subset.csv"),
    "--checkpoint",
    str(PROJECT_ROOT / "outputs" / "checkpoints" / "best_resnet50_awa2.pt"),
    "--figure-dir",
    str(FIG_DIR / "stress_deletion_insertion"),
    "--docs-output-dir",
    str(DOC_ASSET_DIR),
    "--report-output",
    str(REPORT_DIR / "stress_deletion_insertion_curves.csv"),
    "--max-images",
    "4",
    "--ig-steps",
    "16",
    "--ig-internal-batch-size",
    "4",
    "--curve-steps",
    "10",
]
subprocess.run(stress_curve_cmd, cwd=PROJECT_ROOT, check=True)


Generated assets:

- `outputs/figures/blog_metric_summary.png`
- `outputs/figures/blog_case_study_summary.png`
- `outputs/figures/blog_concept_validation_summary.png`
- `outputs/figures/xai_method_comparison_notebook.png`
- `outputs/figures/xai_incorrect_examples_notebook.png`
- `docs/assets/xai-report/metric-summary.png`
- `docs/assets/xai-report/case-summary.png`
- `docs/assets/xai-report/concept-validation-summary.png`
- `docs/assets/xai-report/correct-attributions.png`
- refreshed `incorrect-attributions.png` and the real ResNet forward-inspection figure `resnet-forward.png` in `docs/assets/xai-report/`
- refreshed `background-perturbations.png`, `stress-gradcam.png`, `stress-integrated-gradients.png`, and `saliency-stress-comparison.png` in `docs/assets/xai-report/`
- refreshed `advanced-gradcam-deletion.png` and `advanced-ig-deletion.png` in `docs/assets/xai-report/`
- refreshed the five `error-*` target-contrast, margin, deletion, and CBM evidence figures
- `docs/assets/xai-report/cbm-real-error-case-1.png`
- `docs/assets/xai-report/cbm-real-error-case-2.png`
- `outputs/figures/phase7_spurious_tcav_pairs.png`
- `outputs/figures/phase7_tcav_bridge_concepts.png`
- `docs/assets/xai-report/tcav-spurious-pairs.png`
- `docs/assets/xai-report/tcav-bridge-concepts.png`
- `outputs/figures/stress_metric_dashboard_gaussian-noise.png`
- `outputs/figures/stress_metric_gaussian-noise_case.png`
- `outputs/figures/stress_metric_dashboard_colour-shift.png`
- `outputs/figures/stress_metric_colour-shift_case.png`
- `outputs/figures/stress_metric_dashboard_background-replacement.png`
- `outputs/figures/stress_metric_background-replacement_case.png`
- `outputs/figures/stress_deletion_insertion/gaussian-noise_gradcam_deletion_insertion.png`
- `outputs/figures/stress_deletion_insertion/gaussian-noise_integrated-gradients_deletion_insertion.png`
- `outputs/figures/stress_deletion_insertion/colour-shift_gradcam_deletion_insertion.png`
- `outputs/figures/stress_deletion_insertion/colour-shift_integrated-gradients_deletion_insertion.png`
- `outputs/figures/stress_deletion_insertion/background-replacement_gradcam_deletion_insertion.png`
- `outputs/figures/stress_deletion_insertion/background-replacement_integrated-gradients_deletion_insertion.png`
- `outputs/reports/stress_metric_dashboard_summary.csv`
- `outputs/reports/stress_deletion_insertion_curves.csv`
- `docs/assets/xai-report/stress-metric-gaussian-noise.png`
- `docs/assets/xai-report/stress-curve-gaussian-noise-gradcam.png`
- `docs/assets/xai-report/stress-curve-gaussian-noise-integrated-gradients.png`
- `docs/assets/xai-report/stress-metric-gaussian-noise-case.png`
- `docs/assets/xai-report/stress-metric-colour-shift.png`
- `docs/assets/xai-report/stress-curve-colour-shift-gradcam.png`
- `docs/assets/xai-report/stress-curve-colour-shift-integrated-gradients.png`
- `docs/assets/xai-report/stress-metric-colour-shift-case.png`
- `docs/assets/xai-report/stress-metric-background-replacement.png`
- `docs/assets/xai-report/stress-curve-background-replacement-gradcam.png`
- `docs/assets/xai-report/stress-curve-background-replacement-integrated-gradients.png`
- `docs/assets/xai-report/stress-metric-background-replacement-case.png`
